# hoopstats — train the bib reader (Colab)

Milestone 2, step 3 (`doc/plan-milestone2.md`). Reads bib numbers from torso crops.

1. **Crops**: for each dev video, torso crops of every player box in the TrackID3x3 ground truth, labeled with
   the player's bib number (`hoopstats bib-crops`). Saved to Drive, so they are made only once.
2. **Leave-one-video-out**: for each video, train on the others and evaluate on it
   (`scripts/train_bib_reader.py`): crop accuracy and a vote over 3-second windows (like a track fragment).
3. **Final model** on all dev videos → `bib_final.pt`.

**Before running**

| Drive path | What | How |
|---|---|---|
| `MyDrive/hoopstats/hoopstats_bundle.zip` | code, labels, models, dataset ground truth | on the Mac: `uv run python scripts/make_colab_bundle.py` → upload `data/colab/hoopstats_bundle.zip` (**new bundle needed**: it now includes the ground truth) |
| `MyDrive/hoopstats/videos/IMG_0104.MOV`, … | dev videos | already there |

**Runtime:** A100 (L4/T4 also fine: the model is small). Run all. Everything is saved to
`MyDrive/hoopstats/bib/` as it finishes; if the session disconnects, run all again and finished steps are skipped.
When done: Runtime → Disconnect and delete runtime.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc

## Parameters

In [ ]:
DRIVE_DIR   = "/content/drive/MyDrive/hoopstats"
VIDEOS      = ["IMG_0104", "IMG_0105", "IMG_0106", "IMG_0107", "IMG_0108"]   # dev videos only (never test videos)
EVERY       = 8               # crop every 8th frame (~3.7 per second per player)
MODEL       = "yolo11n-cls.pt"
IMGSZ       = 128
EPOCHS      = 20
BATCH       = 256
FINAL       = True            # also train the final model on all videos
REDO        = False           # True: redo folds whose results are already on Drive
ULTRALYTICS = "8.4.163"       # same version as the project's uv.lock

## Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil, subprocess, sys, zipfile, json, time
from pathlib import Path

bundle = Path(DRIVE_DIR) / "hoopstats_bundle.zip"
assert bundle.exists(), f"upload the bundle first: {bundle}"
ROOT = Path("/content/hoopstats")
shutil.rmtree(ROOT, ignore_errors=True)
with zipfile.ZipFile(bundle) as z:
    z.extractall("/content")
os.chdir(ROOT)
assert (ROOT / "data/trackid3x3_repo/ground_truth/Outdoor/MOT" / f"{VIDEOS[0]}.txt").exists(), \
    "this bundle has no dataset ground truth: rebuild it with scripts/make_colab_bundle.py"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS}"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "."], check=True)   # torch: Colab's own
# this kernel started before the install: import the package from src/ directly
# (otherwise "hoopstats" resolves to the /content/hoopstats folder as an empty namespace package)
sys.path.insert(0, str(ROOT / "src"))
for m in [m for m in sys.modules if m == "hoopstats" or m.startswith("hoopstats.")]:
    del sys.modules[m]

OUT = Path(DRIVE_DIR) / "bib"
(OUT / "crops").mkdir(parents=True, exist_ok=True)
(OUT / "results").mkdir(parents=True, exist_ok=True)

def run(cmd):
    print("$", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=ROOT)
    for line in p.stdout:
        if "it/s" not in line and "\x1b[K" not in line:     # skip progress bars
            print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(cmd)}")

## 1. Crops (once per video, kept on Drive)

In [ ]:
CROPS = ROOT / "data" / "bib_crops"
for v in VIDEOS:
    z = OUT / "crops" / f"{v}.zip"
    if not z.exists():
        src = Path(DRIVE_DIR) / "videos" / f"{v}.MOV"
        assert src.exists(), f"missing {src}"
        local = ROOT / "data" / "outdoor" / f"{v}.MOV"
        local.parent.mkdir(parents=True, exist_ok=True)
        t = time.time(); shutil.copy(src, local); print(f"== {v}: copied in {time.time() - t:.0f} s", flush=True)
        run(["hoopstats", "bib-crops", str(local), "--every", str(EVERY), "--out", str(CROPS)])
        shutil.make_archive(str(z.with_suffix("")), "zip", CROPS, v)
        local.unlink()
    elif not (CROPS / v / "index.csv").exists():
        with zipfile.ZipFile(z) as f:
            f.extractall(CROPS)          # the zip holds <video>/...
    n = sum(1 for _ in open(CROPS / v / "index.csv")) - 1
    print(f"== {v}: {n} crops")

## 2. Leave-one-video-out

In [ ]:
RUNS = Path("/content/runs/bib")
for held in VIDEOS:
    if (OUT / "results" / f"metrics_{held}.json").exists() and not REDO:
        print(f"== {held}: already on Drive, skipping")
        continue
    t = time.time()
    run([sys.executable, "scripts/train_bib_reader.py", "--crops", str(CROPS), "--videos", *VIDEOS, "--held", held,
         "--out", str(RUNS), "--model", MODEL, "--imgsz", str(IMGSZ), "--epochs", str(EPOCHS), "--batch", str(BATCH)])
    for f in (f"preds_{held}.csv", f"metrics_{held}.json", f"bib_held_{held}.pt"):
        shutil.copy(RUNS / f, OUT / "results" / f)
    print(f"== {held}: done in {(time.time() - t) / 60:.1f} min, saved to {OUT / 'results'}")

## 3. Final model (all dev videos)

In [ ]:
if FINAL and (REDO or not (OUT / "results" / "bib_final.pt").exists()):
    run([sys.executable, "scripts/train_bib_reader.py", "--crops", str(CROPS), "--videos", *VIDEOS, "--held", "none",
         "--out", str(RUNS), "--model", MODEL, "--imgsz", str(IMGSZ), "--epochs", str(EPOCHS), "--batch", str(BATCH)])
    shutil.copy(RUNS / "bib_final.pt", OUT / "results" / "bib_final.pt")
print(sorted(p.name for p in (OUT / "results").iterdir()))

## Summary

In [ ]:
import csv
import numpy as np
from hoopstats.training.bib_eval import evaluate

keys = ["crop_acc_clean", "crop_acc_conf0.8", "crop_coverage_conf0.8", "window_acc_free", "window_acc_roster",
        "window_undecided_roster", "windows"]
print("video      " + "  ".join(f"{k:>22}" for k in keys))
rows_all, probs_all, classes = [], [], None
for v in VIDEOS:
    p = OUT / "results" / f"metrics_{v}.json"
    if not p.exists():
        continue
    m = json.loads(p.read_text())
    print(f"{v}  " + "  ".join(f"{str(m.get(k)):>22}" for k in keys))
    with open(OUT / "results" / f"preds_{v}.csv", newline="") as fh:
        rd = csv.DictReader(fh)
        cls = [c for c in rd.fieldnames if c.startswith("b")]
        classes = classes or cls
        for r in rd:
            rows_all.append(r)
            probs_all.append([float(r.get(c, 0) or 0) for c in classes])
if rows_all:
    pooled = evaluate(rows_all, np.array(probs_all), classes)
    print("\nALL       " + "  ".join(f"{str(pooled.get(k)):>22}" for k in keys))
    print("per number (roster vote):", pooled["roster_acc_per_number"])

## Bring back to the Mac

Download `MyDrive/hoopstats/bib/results/` (a few MB: predictions, metrics, weights) and put it at
`~/Code/basketball/data/bib_results/`. Then tell Claude — the next step (identity per track, step 4) starts from there.